# Task 3 — Train Your First Honest Baselines

**Objective:** Compare an always-stays DummyClassifier with balanced Logistic Regression using the same stratified 5-fold cross-validation splits. A depth-4 balanced Decision Tree is included as the requested learning comparison.

**Important:** `left_within_12m` is the target. Identifier, date, source, and post-outcome columns are excluded. Every imputer, scaler, and encoder is fitted inside each cross-validation training fold to prevent preprocessing leakage.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=FutureWarning)

# Works when launched from the repository root or from notebooks/.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"

ATTRITION_PATH = DATA_DIR / "attrition_dataset.csv"
ATTENDANCE_PATH = DATA_DIR / "attendance_monthly_features.csv"
PAYROLL_PATH = DATA_DIR / "payroll_monthly_features.csv"
BENCHMARK_PATH = DATA_DIR / "baseline_benchmarks.csv"

for path in [ATTRITION_PATH, ATTENDANCE_PATH, PAYROLL_PATH, BENCHMARK_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}. Put the supplied CSV files in the repository's data/ folder."
        )

print("Project root:", ROOT)
print("Dataset files found.")

Project root: /workspace
Dataset files found.


## 1. Load the data and build the feature table

Reuse Task 2's feature-building function so attendance and payroll are aggregated separately and only records available by the employee snapshot date are included.

In [2]:
# Import the Task 2 feature builder from src/features/build_features.py
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.features.build_features import build_features

employees = pd.read_csv(ATTRITION_PATH)
attendance = pd.read_csv(ATTENDANCE_PATH)
payroll = pd.read_csv(PAYROLL_PATH)
benchmarks = pd.read_csv(BENCHMARK_PATH)

df = build_features(employees, attendance, payroll)

TARGET = "left_within_12m"
if TARGET not in df.columns:
    raise ValueError(f"Target column {TARGET!r} is missing.")
df = df.loc[df[TARGET].notna()].copy()
y = pd.to_numeric(df[TARGET], errors="raise").astype(int)

print(f"Labelled rows: {len(df):,}")
print(f"Leavers: {int(y.sum()):,} ({y.mean():.2%})")
print(f"Stayed: {int((y == 0).sum()):,} ({(y == 0).mean():.2%})")
display(y.value_counts().rename(index={0: "Stayed (0)", 1: "Left (1)"}).to_frame("rows"))

Labelled rows: 2,389
Leavers: 349 (14.61%)
Stayed: 2,040 (85.39%)


,rows
left_within_12m,
Stayed (0),2040
Left (1),349


## 2. Define safe predictors and preprocessing

The target and all known identifiers, dates, source markers, and post-outcome fields are excluded. Remaining safe employee features—including Task 2's engineered features—are passed through a scikit-learn preprocessing pipeline. Numeric columns use median imputation and scaling; categorical columns use most-frequent imputation and one-hot encoding. Unknown categories are ignored.

All preprocessing is kept inside each model pipeline, so it is fitted only on the training fold during cross-validation.

In [3]:
FORBIDDEN_COLUMNS = {
    "row_id",
    "employee_id",
    "observation_date",
    TARGET,
    "voluntary_left_within_12m",
    "source",
    "exit_date",
    "resignation_date",
    "exit_reason",
    "exit_interview_done",
    "fnf_settlement_amount",
}

# Remove target/leakage columns, then discard columns that are entirely missing.
X = df.drop(columns=[c for c in FORBIDDEN_COLUMNS if c in df.columns]).copy()
all_missing = [c for c in X.columns if X[c].isna().all()]
if all_missing:
    X = X.drop(columns=all_missing)

# Dates are not model inputs. Convert any remaining date-like columns to text only if safe;
# the known observation date has already been excluded above.
date_like = [c for c in X.columns if pd.api.types.is_datetime64_any_dtype(X[c])]
if date_like:
    X = X.drop(columns=date_like)

numeric_columns = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_columns = [c for c in X.columns if c not in numeric_columns]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_columns),
        ("categorical", categorical_pipeline, categorical_columns),
    ],
    remainder="drop",
)

print(f"Safe predictor columns: {X.shape[1]}")
print(f"Numeric columns: {len(numeric_columns)}")
print(f"Categorical columns: {len(categorical_columns)}")
print("Forbidden columns present in X:", sorted(FORBIDDEN_COLUMNS.intersection(X.columns)))
assert not FORBIDDEN_COLUMNS.intersection(X.columns)
assert len(X) == len(y)

Safe predictor columns: 45
Numeric columns: 39
Categorical columns: 6
Forbidden columns present in X: []


## 3. Run honest 5-fold cross-validation

The same `StratifiedKFold` configuration is reused for every model. `cross_val_predict` generates an out-of-fold probability for each row. The DummyClassifier always predicts the majority class; Logistic Regression uses `class_weight="balanced"` to give the minority class more influence. A depth-4 balanced decision tree is included for comparison.

In [4]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
THRESHOLD = 0.50

models = {
    "Dummy (always stays)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression (balanced)": Pipeline([
        ("preprocess", clone(preprocessor)),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=3000,
            random_state=42,
        )),
    ]),
    "Decision Tree (depth 4, balanced)": Pipeline([
        ("preprocess", clone(preprocessor)),
        ("model", DecisionTreeClassifier(
            max_depth=4,
            class_weight="balanced",
            random_state=42,
        )),
    ]),
}

def calculate_metrics(y_true, probabilities, threshold=THRESHOLD):
    predictions = (probabilities >= threshold).astype(int)
    return {
        "roc_auc_mean": roc_auc_score(y_true, probabilities),
        "pr_auc_mean": average_precision_score(y_true, probabilities),
        "precision_mean": precision_score(y_true, predictions, zero_division=0),
        "recall_mean": recall_score(y_true, predictions, zero_division=0),
        "f1_mean": f1_score(y_true, predictions, zero_division=0),
        "accuracy_mean": accuracy_score(y_true, predictions),
    }

oof_probabilities = {}
rows = []

for model_name, model in models.items():
    print(f"Cross-validating: {model_name}")
    # Same CV splitter (same 5 folds) for each model.
    probabilities = cross_val_predict(
        model, X, y, cv=CV, method="predict_proba", n_jobs=None
    )[:, 1]
    oof_probabilities[model_name] = probabilities
    metric_row = {"model": model_name, **calculate_metrics(y, probabilities)}
    rows.append(metric_row)

results = pd.DataFrame(rows)
display(results.round(3))

Cross-validating: Dummy (always stays)
Cross-validating: Logistic Regression (balanced)
Cross-validating: Decision Tree (depth 4, balanced)


,model,roc_auc_mean,pr_auc_mean,precision_mean,recall_mean,f1_mean,accuracy_mean
0,Dummy (always stays),0.500,0.146,0.000,0.000,0.000,0.854
1,Logistic Regression (balanced),0.699,0.297,0.256,0.596,0.359,0.689
2,"Decision Tree (depth 4, balanced)",0.667,0.264,0.240,0.562,0.337,0.677


### Task 3 — Benchmark Comparison

1. The DummyClassifier achieved a ROC-AUC of 0.500, as expected from a model that always predicts the majority class.
2. Its accuracy was 0.854, but its recall for employees who leave was zero, showing why accuracy alone is misleading for this imbalanced dataset.
3. Balanced Logistic Regression achieved a ROC-AUC of 0.699, which is 0.014 below the benchmark of approximately 0.713.
4. This difference is small and does not by itself indicate a problem, although cross-validation variability should also be considered.
5. The Decision Tree achieved a ROC-AUC of 0.667, below Logistic Regression, so it did not improve the baseline in this experiment.

## 4. Inspect the majority-class baseline

Accuracy alone is misleading for this imbalanced dataset. The dummy model can be around 85% accurate simply by predicting “stayed” for everyone, while recall for leavers is zero.

In [5]:
dummy_probabilities = oof_probabilities["Dummy (always stays)"]
dummy_predictions = (dummy_probabilities >= THRESHOLD).astype(int)
dummy_matrix = confusion_matrix(y, dummy_predictions, labels=[0, 1])

display(pd.DataFrame(
    dummy_matrix,
    index=["Actual stayed", "Actual left"],
    columns=["Predicted stayed", "Predicted left"],
))
print(f"Dummy accuracy: {accuracy_score(y, dummy_predictions):.3f}")
print(f"Dummy recall for leavers: {recall_score(y, dummy_predictions, zero_division=0):.3f}")
print(f"Dummy ROC-AUC: {roc_auc_score(y, dummy_probabilities):.3f}")

,Predicted stayed,Predicted left
Actual stayed,2040,0
Actual left,349,0


Dummy accuracy: 0.854
Dummy recall for leavers: 0.000
Dummy ROC-AUC: 0.500


## 5. Compare results with the supplied benchmark sheet

The benchmark is a reference, not a score that should be forced. Compare the measured ROC-AUC, PR-AUC, precision, recall, F1 and accuracy with the provided values. Differences can arise from feature selection, preprocessing, library versions, and cross-validation details.

In [6]:
display(benchmarks)

benchmark_lookup = benchmarks.set_index("model")
comparison_rows = []
benchmark_name_map = {
    "Dummy (always stays)": "Dummy (always stays)",
    "Logistic Regression (balanced)": "Logistic Regression (class_weight=balanced)",
    "Decision Tree (depth 4, balanced)": "Decision Tree (depth 4, balanced)",
}

for _, row in results.iterrows():
    benchmark_name = benchmark_name_map[row["model"]]
    if benchmark_name in benchmark_lookup.index:
        reference = benchmark_lookup.loc[benchmark_name]
        comparison_rows.append({
            "model": row["model"],
            "measured ROC-AUC": row["roc_auc_mean"],
            "benchmark ROC-AUC": reference["roc_auc_mean"],
            "ROC-AUC difference": row["roc_auc_mean"] - reference["roc_auc_mean"],
            "measured PR-AUC": row["pr_auc_mean"],
            "benchmark PR-AUC": reference["pr_auc_mean"],
            "measured precision": row["precision_mean"],
            "benchmark precision": reference["precision_mean"],
            "measured recall": row["recall_mean"],
            "benchmark recall": reference["recall_mean"],
            "measured F1": row["f1_mean"],
            "benchmark F1": reference["f1_mean"],
            "measured accuracy": row["accuracy_mean"],
            "benchmark accuracy": reference["accuracy_mean"],
        })

comparison = pd.DataFrame(comparison_rows)
display(comparison.round(3))

,model,roc_auc_mean,roc_auc_sd,pr_auc_mean,precision_mean,recall_mean,f1_mean,accuracy_mean,comment
0,Dummy (always stays),0.500,0.000,0.146,0.000,0.000,0.000,0.854,Looks 85% accurate yet catches nobody: why acc...
1,Logistic Regression (class_weight=balanced),0.713,0.010,0.332,0.260,0.630,0.368,0.683,Clean features only (reference target)
2,"Decision Tree (depth 4, balanced)",0.650,0.019,0.256,0.234,0.524,0.322,0.676,Clean features only (reference target)
3,"Random Forest (300 trees, balanced)",0.715,0.018,0.336,0.523,0.089,0.151,0.854,Clean features only (reference target)
4,LEAKY Logistic Regression (adds exit information),1.000,0.000,1.000,1.000,1.000,1.000,1.000,Too good to be true: this is data leakage


,model,measured ROC-AUC,benchmark ROC-AUC,ROC-AUC difference,measured PR-AUC,benchmark PR-AUC,measured precision,benchmark precision,measured recall,benchmark recall,measured F1,benchmark F1,measured accuracy,benchmark accuracy
0,Dummy (always stays),0.500,0.500,0.000,0.146,0.146,0.000,0.000,0.000,0.000,0.000,0.000,0.854,0.854
1,Logistic Regression (balanced),0.699,0.713,-0.014,0.297,0.332,0.256,0.260,0.596,0.630,0.359,0.368,0.689,0.683
2,"Decision Tree (depth 4, balanced)",0.667,0.650,0.017,0.264,0.256,0.240,0.234,0.562,0.524,0.337,0.322,0.677,0.676


The DummyClassifier achieved a ROC-AUC of 0.500, as expected from a model that always predicts the majority class. Its accuracy was 0.854, but its recall for employees who leave was zero, showing why accuracy alone is misleading for this imbalanced dataset. Balanced Logistic Regression achieved a ROC-AUC of 0.699, which is 0.014 below the benchmark of approximately 0.713. This difference is small and does not by itself indicate a problem, although cross-validation variability should also be considered. The Decision Tree achieved a ROC-AUC of 0.667, below Logistic Regression, so it did not improve the baseline in this experiment.

## 6. Five-sentence comparison note

This cell generates a concise evidence-based note from the measured results. Read it before submission and ensure it accurately describes the run.

In [7]:
log_row = results.loc[results["model"] == "Logistic Regression (balanced)"].iloc[0]
dummy_row = results.loc[results["model"] == "Dummy (always stays)"].iloc[0]
log_benchmark = benchmark_lookup.loc["Logistic Regression (class_weight=balanced)"]
auc_gap = log_row["roc_auc_mean"] - log_benchmark["roc_auc_mean"]
direction = "above" if auc_gap >= 0 else "below"

sentences = [
    f"The target is imbalanced: {y.mean():.1%} of labelled snapshots are leavers, so accuracy alone is not sufficient.",
    f"The always-stays Dummy baseline achieved ROC-AUC {dummy_row['roc_auc_mean']:.3f} and recall {dummy_row['recall_mean']:.3f}; it misses every leaver at the 0.50 threshold.",
    f"Balanced Logistic Regression achieved ROC-AUC {log_row['roc_auc_mean']:.3f}, PR-AUC {log_row['pr_auc_mean']:.3f}, recall {log_row['recall_mean']:.3f}, and F1 {log_row['f1_mean']:.3f}.",
    f"Its ROC-AUC is {abs(auc_gap):.3f} {direction} the supplied benchmark of {log_benchmark['roc_auc_mean']:.3f}; this difference is plausible because the reference and local run may use different feature choices or preprocessing details.",
    "The result is below the reference target of about 0.71, so I would verify feature parity and leakage exclusions before trying more complex models rather than claiming the model is better than the benchmark."
]
comparison_note = "\n".join(f"{i+1}. {sentence}" for i, sentence in enumerate(sentences))
print(comparison_note)

1. The target is imbalanced: 14.6% of labelled snapshots are leavers, so accuracy alone is not sufficient.
2. The always-stays Dummy baseline achieved ROC-AUC 0.500 and recall 0.000; it misses every leaver at the 0.50 threshold.
3. Balanced Logistic Regression achieved ROC-AUC 0.699, PR-AUC 0.297, recall 0.596, and F1 0.359.
4. Its ROC-AUC is 0.014 below the supplied benchmark of 0.713; this difference is plausible because the reference and local run may use different feature choices or preprocessing details.
5. The result is below the reference target of about 0.71, so I would verify feature parity and leakage exclusions before trying more complex models rather than claiming the model is better than the benchmark.


## 7. Example Low / Medium / High risk bands

These are illustrative score bands for inspection, not calibrated or approved business thresholds. Task 4 should define the proper evaluation metrics and risk bands before operational use.

In [8]:
def risk_bands(probabilities):
    probabilities = np.asarray(probabilities, dtype=float)
    return np.select(
        [probabilities >= 0.70, probabilities >= 0.30],
        ["High", "Medium"],
        default="Low",
    )

risk_summary = pd.DataFrame({
    "actual_left": y.to_numpy(),
    "oof_probability": oof_probabilities["Logistic Regression (balanced)"],
    "risk_band": risk_bands(oof_probabilities["Logistic Regression (balanced)"]),
})
display(risk_summary.groupby("risk_band", observed=False).agg(
    employees=("actual_left", "size"),
    observed_leavers=("actual_left", "sum"),
    mean_predicted_probability=("oof_probability", "mean"),
).sort_values("mean_predicted_probability", ascending=False).round(3))

print("Risk bands are illustrative only; they are not calibrated or validated business thresholds.")

,employees,observed_leavers,mean_predicted_probability
risk_band,,,
High,350,115,0.821
Medium,1043,160,0.489
Low,996,74,0.160


Risk bands are illustrative only; they are not calibrated or validated business thresholds.


## 8. Final checks

- The target is never included in `X`.
- Known IDs, dates, source markers, and post-outcome fields are excluded.
- Preprocessing is inside each model pipeline and is fitted separately within each CV fold.
- All models use the same stratified five-fold split configuration (`shuffle=True`, `random_state=42`).
- Out-of-fold probabilities are used for ROC-AUC and PR-AUC.
- A high score should trigger a leakage review; it is not automatically evidence of a better model.